# 03 - SVAR and Inference

Notebook goals:
- Document recursive ordering rationale
- Estimate the recursive SVAR (via VAR + Cholesky identification)
- Visualize IRF with confidence bands
- Visualize FEVD and quantify policy-shock contribution to NASDAQ variance

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from nasdaq_svar.config import load_config
from nasdaq_svar.pipeline import run_pipeline
from nasdaq_svar.presentation import build_chapter3_conclusion
from nasdaq_svar.stage2_svar import run_stage2_recursive_svar

plt.style.use("seaborn-v0_8-whitegrid")
cfg = load_config(PROJECT_ROOT / "configs/default.yaml")

## Cholesky ordering used

1. `INDPRO` (slow-moving real activity)
2. `CPIAUCNS` (sticky-price inflation)
3. `FEDFUNDS` (policy instrument)
4. `NASDAQ_SA` (asset price reacts contemporaneously)

This aligns with the recursive chain in the research outline.

In [ ]:
stage2_path = PROJECT_ROOT / "data/processed/stage2_transformed_dataset.csv"
if not stage2_path.exists():
    _ = run_pipeline(config_path=PROJECT_ROOT / "configs/default.yaml", project_root=PROJECT_ROOT)

stage2_data = pd.read_csv(stage2_path, index_col=0, parse_dates=True)
result = run_stage2_recursive_svar(stage2_data, cfg["stage2"])
print("Selected lag:", result.selected_lag)
result.lag_selection

In [ ]:
irf = result.irf_table.copy()
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(irf["horizon"], irf["orth_irf"], color="#1f77b4", lw=2, label="IRF")
if {"lower", "upper"}.issubset(irf.columns):
    conf_label = f"{int(result.irf_confidence_level * 100)}% MC band"
    ax.fill_between(irf["horizon"], irf["lower"], irf["upper"], color="#1f77b4", alpha=0.2, label=conf_label)
ax.axhline(0.0, color="black", lw=1.0, linestyle="--")
ax.set_xlabel("Horizon (months)")
ax.set_ylabel("NASDAQ_SA response")
ax.set_title("Orthogonalized IRF: NASDAQ_SA to FEDFUNDS shock")
ax.legend(frameon=False)
fig.tight_layout()
irf.head()

In [ ]:
fevd = result.fevd_table.copy()
shock_cols = [c for c in fevd.columns if c.startswith("shock_")]

fig, ax = plt.subplots(figsize=(10, 5))
for col in shock_cols:
    ax.plot(fevd["horizon"], fevd[col], lw=1.8, label=col)
ax.set_xlabel("Horizon (months)")
ax.set_ylabel("Variance share")
ax.set_title("FEVD of NASDAQ_SA")
ax.set_ylim(0, 1)
ax.legend(frameon=False, ncols=2)
fig.tight_layout()

In [ ]:
policy_col = "shock_FEDFUNDS"
share_12 = fevd.loc[fevd["horizon"] == 12, policy_col].iloc[0]
share_24 = fevd.loc[fevd["horizon"] == 24, policy_col].iloc[0]
print(f"FEVD policy-share at 12 months: {share_12:.4%}")
print(f"FEVD policy-share at 24 months: {share_24:.4%}")

## Slide-Ready Conclusion

In [ ]:
chapter3_text = build_chapter3_conclusion(
    irf_table=result.irf_table,
    fevd_table=result.fevd_table,
    response_variable=cfg["stage2"]["equity_response_variable"],
    shock_variable=cfg["stage2"]["policy_shock_variable"],
    confidence_level=result.irf_confidence_level,
)
display(Markdown(chapter3_text))